# NBA 3.1 - 11 Adoption speed features (per claims cohort)

Upgrade of NBA 3.0 `03_Feature_Engineering/03_HCP_Level_Features/11_adoption_features_inv_far` and of the earlier 3.1 draft `03_HCP_Level_Features/Archive/11_adoption_features`. Spec: `Feature Analysis/92_competitor_comorbidity_nba31_readiness_plan.md` Phase 3 (AD1-AD3) and `04_adoption_speed_features_nba31_changes.md`. Plain-language summary: `93_competitor_comorbidity_plan_in_simple_words.md`.

**What changes**

| NBA 3.0 | NBA 3.1 |
| --- | --- |
| `TIME_TO_ADOPT_INVOKANA`, `TIME_TO_ADOPT_FARXIGA`, one static row per HCP (look-ahead leakage) | 14 columns per HCP x claims cohort for 5 drug groups (AD1) + `ENTRESTO_BRAND_ADOPTER_FLAG` (AD3) = 15 columns |
| 3.0 universe and volume | 3.1 universe without call-only HCPs (294,727) and `hcp_level_model/dev_pipeline` paths |
| CKD Dx from `Dx_03` (almost no stage 3 claims) | CKD Dx from the NBA 3.1 Dx event list `dx_input_claims_all_events_features_input` (CKD + ESRD codes of `09_claim_comorbidity` + `D63.1`), V2 fix |

Changes vs the Archive draft: new universe, `dev_pipeline` input/output paths, `ENTRESTO_PRODUCTS` (brand + generic names, AD2) and the brand-adopter flag (AD3).

| # | Column | Definition |
| --- | --- | --- |
| 1 | `MIN_DAYS_TO_PRESCRIBE_{DRUG}_POST_CKD_DIAGNOSIS` | NBA 3.0 `TIME_TO_ADOPT_{DRUG}` renamed. Min over the HCP's patients of days from the patient's first CKD Dx **by this HCP** to the patient's first Rx of the drug from this HCP (negatives dropped) |
| 2 | `HCP_TIME_TO_ADOPT_{DRUG}` | Days from FDA approval to the HCP's first Rx of the drug (any patient). Not built for `STEROIDAL_MRA` |
| 3 | `MIN_DAYS_TO_PRESCRIBE_{DRUG}_POST_FIRST_CKD_DIAGNOSIS_ANY_HCP` | Same as #1, but from the patient's first CKD Dx by **any** provider |
| 4 | `ENTRESTO_BRAND_ADOPTER_FLAG` | 1 if the HCP's first Entresto Rx is before the Entresto LOE date (2025-07-01) and on or before the cohort cutoff, else 0 |

`{DRUG}` = `INVOKANA`, `FARXIGA`, `JARDIANCE`, `STEROIDAL_MRA` (spironolactone + eplerenone), `ENTRESTO`.

**Business rules** (unchanged from NBA 3.0 for #1): Farxiga Dx and Rx on or after 2021-04-01, Invokana on or after 2019-10-01, other groups full history. Rejected claims included. Empty when no qualifying patient / Rx by the cutoff (empty and 0 are different for the model). Each cohort uses claims up to the end of its `lookback_end_date` month.

**CKD source (V2 fix, 2026-11-18):** the CKD diagnosis dates come from the Dx event list instead of `Dx_03`: 20,200,200 CKD patients vs 16,634,749 in `Dx_03`, including 11,685,734 with a stage 3 code; only 54,697 `Dx_03` patients are not covered. In c28 the CKD-based columns gained 0.8 to 4.8 points of fill rate.

**Entresto generic (V6):** `Rx_01_new` was re-pulled on 2026-11-17 with generic `SACUBITRIL/VALSARTAN` (first claim 2024-11-27). The Entresto group uses brand + generic; `ENTRESTO_BRAND_ADOPTER_FLAG` uses brand only.

Output: `hcp_level_model/dev_pipeline/notebook_output/hcp_features/adoption_features` (Delta). Widget `TEST_HCP_SAMPLE` > 0 writes to `adoption_features_test`.

## Phase 0 validation status (plan 92, Section 3), checked 2026-11-17

**Score: 7 of 10 met, 2 partly met, 1 not met** (V2 met after the CKD source fix on 2026-11-18). Checks that matter for this notebook are in **bold**.

| # | Validation | Result | Status |
| --- | --- | --- | --- |
| V1 | Indication coverage in LAAD | SGLT2 91.5-93.4%, Entresto 100%, steroidal MRA 44.1-49.3% with an indication group; stable | Met (LRx only, not used here) |
| **V2** | **CKD stage 3 codes present** | **`Dx_03` has only 25,167 stage-3 patients; 3,144,676 stage-3 patients had no `Dx_03` CKD claim. Fixed 2026-11-18: the CKD source is now the Dx event list (20,200,200 CKD patients, 11,685,734 with stage 3; 54,697 `Dx_03` patients not covered)** | **Met (fixed).** c28 fill rates rose, e.g. Farxiga same-HCP 43.28% to 46.83%, steroidal MRA 46.8% to 51.64% |
| V3 | HF LVEF >= 40% proxy sign-off | Medical review not done | Not met (not used here) |
| V4 | CKD indication groups in LRx | CKD, T2D + CKD, T2D + CKD + HF, CKD + HF | Met |
| V5 | Farxiga brand vs generic labels | Generic LRx NBRx from 2023-11-03, rising from Jan 2026 | Partly met (not used here; `Rx_01_new` has one FARXIGA name) |
| **V6** | **Entresto generic** | **`Rx_01_new`: SACUBITRIL/VALSARTAN 7,750,567 rows (+2 rows SACUBITRIL/VALSART), first claim 2024-11-27. LRx generic first NBRx 2025-01-31, ramp Jul-Aug 2025 (541 to 13,162), confirming the Jul 2025 LOE** | **Met.** Exact LOE day still to confirm with Josh (Q6) |
| V7 | GLP-1 in LRx | 7.01M 12M NBRx | Met (not used here) |
| V8 | Parity vs NBA 3.0 | Proxy only for competitor features | Partly met (not used here) |
| V9 | Rows after each event | Entresto LOE: 14 cohorts | Met |
| **V10** | **Entresto prescriber specialty mix** | **Brand Entresto writers 2023-24 (187,164): PCP 49.1%, NP/PA 31.3%, CARD 16.1%, NEPH 2.6%. Kerendia writers 2023-24 (28,697): PCP 42.1%, NP/PA 29.6%, NEPH 19.0%, CARD 3.4%. 69.9% of Kerendia writers also wrote brand Entresto. Kerendia writer rate: 10.7% for brand Entresto writers vs 8.0% for other HCPs** | **Met (answered).** The brand-adopter flag is a weak target-segment signal: it is cardiology-heavy, while Kerendia is nephrology-heavy |

**For this notebook:** V2, V6 and V10 pass. The CKD-based columns (10 of 15) now use the Dx event list.

In [0]:
import os, sys, json, builtins, importlib, copy
from datetime import datetime as dt
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0 = full universe)")
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))

# Shared constants: hcp_features_config.py in this folder (paths, drug groups, dates)
sys.path.insert(0, os.getcwd())
import hcp_features_config as cfg
importlib.reload(cfg)

VOL31 = cfg.VOL31
COHORTS_DIR = cfg.COHORTS_DIR
HCP_UNIVERSE_PATH = cfg.HCP_UNIVERSE_PATH
QUERY_OUTPUT_31 = cfg.QUERY_OUTPUT_31
HCP_FEATURES_OUT_31 = cfg.HCP_FEATURES_OUT_31
OUTPUT_PATH = f"{HCP_FEATURES_OUT_31}/adoption_features" + ("_test" if TEST_HCP_SAMPLE else "")
assert "kerendia_nba_3_1_volume" in OUTPUT_PATH

# AD2: Entresto brand + generic (generic names are picked up once Rx_01_new is re-pulled with them)
ENTRESTO_BRAND_PRODUCTS = list(cfg.ENTRESTO_BRAND_PRODUCTS)
ENTRESTO_PRODUCTS = list(cfg.ENTRESTO_PRODUCTS)   # brand + LAAD generic names (from 2024-11-27)

DRUG_GROUPS = copy.deepcopy(cfg.RX01_DRUG_GROUPS)   # Rx_01_new PRODUCT_NAME groups
# FDA approval dates for #2 (confirm with Medical). No entry = #2 not built
APPROVAL_DATES = dict(cfg.APPROVAL_DATES)
# NBA 3.0 date floors for #1 / #3 (Dx and Rx on or after the CKD-indication approval)
DATE_FLOORS = dict(cfg.DATE_FLOORS)
ENTRESTO_LOE_DATE = cfg.ENTRESTO_LOE_DATE     # AD3 / EV2; same market event as 36_market_event_features (cfg.MARKET_EVENTS)

def f1(g): return f"MIN_DAYS_TO_PRESCRIBE_{g}_POST_CKD_DIAGNOSIS"
def f2(g): return f"HCP_TIME_TO_ADOPT_{g}"
def f3(g): return f"MIN_DAYS_TO_PRESCRIBE_{g}_POST_FIRST_CKD_DIAGNOSIS_ANY_HCP"
FLAG_COL = "ENTRESTO_BRAND_ADOPTER_FLAG"

DAY_COLS = []
for g in DRUG_GROUPS:
    DAY_COLS.append(f1(g))
    if g in APPROVAL_DATES:
        DAY_COLS.append(f2(g))
    DAY_COLS.append(f3(g))
assert len(DAY_COLS) == 14
FEATURE_COLS = DAY_COLS + [FLAG_COL]
print(len(FEATURE_COLS), "features ->", OUTPUT_PATH)

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert meta["feature_family"] == "claims", meta["feature_family"]
assert len(cohorts) == meta["cohort_count"] and (cohorts["offset_days"] == 0).all()

coh = (spark.createDataFrame(cohorts[["cohort_id", "lookback_end_date"]].astype(str))
       .withColumn("LOOKBACK_END_M", F.to_date("lookback_end_date")).drop("lookback_end_date"))

hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string")).distinct()
if TEST_HCP_SAMPLE:
    hcp_universe = hcp_universe.orderBy(F.xxhash64("PROVIDER_ID")).limit(TEST_HCP_SAMPLE)
hcp_universe = hcp_universe.cache()
n_hcp = hcp_universe.count()
print(f"Claims cohorts: {len(cohorts)} | HCPs {n_hcp:,}")

In [0]:
dx = (spark.read.format("delta").load(f"{QUERY_OUTPUT_31}/Dx_03")
      .select("PROVIDER_ID", "PATIENT_ID", F.to_date("SVC_DT").alias("SVC_DT"))
      .filter(F.col("PROVIDER_ID").isNotNull() & F.col("PATIENT_ID").isNotNull()))
# dx keeps every provider: #3 needs the patient's first CKD date from ANY provider

prod_to_grp = F.create_map(*[x for g, ps in DRUG_GROUPS.items() for p in ps for x in (F.lit(p), F.lit(g))])
all_products = [p for ps in DRUG_GROUPS.values() for p in ps]
rx_raw = (spark.read.format("delta").load(f"{QUERY_OUTPUT_31}/Rx_01_new")
          .withColumn("PRODUCT_NAME", F.upper(F.trim("PRODUCT_NAME")))
          .filter(F.col("PRODUCT_NAME").isin(all_products))
          .join(F.broadcast(hcp_universe), "PROVIDER_ID", "left_semi"))
rx = (rx_raw.select("PROVIDER_ID", "PATIENT_ID", F.to_date("SVC_DT").alias("SVC_DT"), "PRODUCT_NAME",
                    prod_to_grp[F.col("PRODUCT_NAME")].alias("GRP"))
        .filter(F.col("PATIENT_ID").isNotNull()))

rx_summary = (rx.groupBy("GRP", "PRODUCT_NAME").agg(F.count("*").alias("rx_rows"), F.countDistinct("PROVIDER_ID").alias("hcps"),
                                                   F.min("SVC_DT").alias("first_svc_dt"), F.max("SVC_DT").alias("last_svc_dt"))
              .orderBy("GRP", "PRODUCT_NAME"))
display(rx_summary)
found = {(r["GRP"], r["PRODUCT_NAME"]) for r in rx_summary.collect()}
missing = set(DRUG_GROUPS) - {g for g, _ in found}
assert not missing, f"Drug groups missing from Rx_01_new - re-pull with 00_Write_HCP_Inputs_31: {missing}"
print("V6 - Entresto products found in Rx_01_new:", sorted(p for g, p in found if g == "ENTRESTO"))

In [0]:
# CKD diagnoses from the NBA 3.1 Dx event list (V2 fix, 2026-11-18; replaces Dx_03).
# Dx_03 has almost no stage 3 claims (N18.30/31/32); 3.1M stage-3 patients had no Dx_03 CKD claim at all.
# Codes = CKD + ESRD building blocks of 09_claim_comorbidity, plus D63.1 (anemia in CKD), which NBA 3.0 Dx_03 also counted.
# From the latest dated Dx Product Basket (cfg.dx_code_sets: CKD_ANY), not a code list.
_DX_SETS = cfg.dx_code_sets()
CKD_DX_CODES = _DX_SETS["CKD_ANY"]
ST3_CODES = _DX_SETS["ST3"]
print(f"Dx basket {os.path.basename(cfg.product_mapping_path('dx'))}: {len(CKD_DX_CODES)} CKD codes, {len(ST3_CODES)} stage 3")
DX_EVENTS = cfg.DX_EVENTS_PATH
dx = (spark.read.format("delta").load(DX_EVENTS)
      .filter(F.upper(F.trim("DIAGNOSIS_CODE")).isin(CKD_DX_CODES))
      .select("PROVIDER_ID", "PATIENT_ID", F.to_date("SVC_DT").alias("SVC_DT"))
      .filter(F.col("PROVIDER_ID").isNotNull() & F.col("PATIENT_ID").isNotNull()))
# dx keeps every provider: #3 needs the patient's first CKD date from ANY provider

# V2 check vs the old Dx_03 source (Dx_03 is read only for this comparison)
_dx03_pat = spark.read.format("delta").load(f"{QUERY_OUTPUT_31}/Dx_03").select("PATIENT_ID").distinct()
_pat = dx.select("PATIENT_ID").distinct()
_st3 = (spark.read.format("delta").load(DX_EVENTS)
        .filter(F.upper(F.trim("DIAGNOSIS_CODE")).isin(ST3_CODES))
        .select("PATIENT_ID").distinct())
print(f"V2 - CKD patients: event list {_pat.count():,} vs Dx_03 {_dx03_pat.count():,} | "
      f"Dx_03 patients not in the new source {_dx03_pat.join(_pat, 'PATIENT_ID', 'left_anti').count():,} | "
      f"stage-3 patients {_st3.count():,}")

prod_to_grp = F.create_map(*[x for g, ps in DRUG_GROUPS.items() for p in ps for x in (F.lit(p), F.lit(g))])
all_products = [p for ps in DRUG_GROUPS.values() for p in ps]
rx_raw = (spark.read.format("delta").load(f"{QUERY_OUTPUT_31}/Rx_01_new")
          .withColumn("PRODUCT_NAME", F.upper(F.trim("PRODUCT_NAME")))
          .filter(F.col("PRODUCT_NAME").isin(all_products))
          .join(F.broadcast(hcp_universe), "PROVIDER_ID", "left_semi"))
rx = (rx_raw.select("PROVIDER_ID", "PATIENT_ID", F.to_date("SVC_DT").alias("SVC_DT"), "PRODUCT_NAME",
                    prod_to_grp[F.col("PRODUCT_NAME")].alias("GRP"))
        .filter(F.col("PATIENT_ID").isNotNull()))

rx_summary = (rx.groupBy("GRP", "PRODUCT_NAME").agg(F.count("*").alias("rx_rows"), F.countDistinct("PROVIDER_ID").alias("hcps"),
                                                   F.min("SVC_DT").alias("first_svc_dt"), F.max("SVC_DT").alias("last_svc_dt"))
              .orderBy("GRP", "PRODUCT_NAME"))
display(rx_summary)
found = {(r["GRP"], r["PRODUCT_NAME"]) for r in rx_summary.collect()}
missing = set(DRUG_GROUPS) - {g for g, _ in found}
assert not missing, f"Drug groups missing from Rx_01_new - re-pull with 00_Write_HCP_Inputs_31: {missing}"
print("V6 - Entresto products found in Rx_01_new:", sorted(p for g, p in found if g == "ENTRESTO"))

In [0]:
# First dates over the full history. For a cohort with cutoff C, a patient counts when the first Rx is <= C
# (gap >= 0 means the first CKD date is also <= C), which gives the same values as recomputing with SVC_DT <= C.
floor_col = F.create_map(*[x for g, d in DATE_FLOORS.items() for x in (F.lit(g), F.lit(d).cast("date"))])
rx_pair = (rx.withColumn("FLOOR", floor_col[F.col("GRP")])
             .filter(F.col("FLOOR").isNull() | (F.col("SVC_DT") >= F.col("FLOOR")))
             .groupBy("PROVIDER_ID", "PATIENT_ID", "GRP").agg(F.min("SVC_DT").alias("RX_DT")))

floors = sorted({DATE_FLOORS.get(g) for g in DRUG_GROUPS}, key=lambda x: x or "")
gap_parts = []
for fl in floors:
    grps = [g for g in DRUG_GROUPS if DATE_FLOORS.get(g) == fl]
    dx_f = dx if fl is None else dx.filter(F.col("SVC_DT") >= F.lit(fl).cast("date"))
    dx_pair = dx_f.groupBy("PROVIDER_ID", "PATIENT_ID").agg(F.min("SVC_DT").alias("CKD_DT"))
    dx_pat = dx_f.groupBy("PATIENT_ID").agg(F.min("SVC_DT").alias("CKD_DT_ANY"))
    gap_parts.append(rx_pair.filter(F.col("GRP").isin(grps))
                     .join(dx_pair, ["PROVIDER_ID", "PATIENT_ID"], "left")
                     .join(dx_pat, "PATIENT_ID", "left")
                     .withColumn("GAP_SAME", F.datediff("RX_DT", "CKD_DT"))
                     .withColumn("GAP_ANY", F.datediff("RX_DT", "CKD_DT_ANY")))
gaps = gap_parts[0]
for p in gap_parts[1:]:
    gaps = gaps.unionByName(p)

# Smallest non-negative gap per HCP, group and Rx month (cohort cutoffs are month ends)
gaps_m = (gaps.withColumn("RX_M", F.trunc("RX_DT", "month"))
          .groupBy("PROVIDER_ID", "GRP", "RX_M")
          .agg(F.min(F.when(F.col("GAP_SAME") >= 0, F.col("GAP_SAME"))).alias("GAP_SAME"),
               F.min(F.when(F.col("GAP_ANY") >= 0, F.col("GAP_ANY"))).alias("GAP_ANY"))
          .filter(F.col("GAP_SAME").isNotNull() | F.col("GAP_ANY").isNotNull()))

# #2: HCP's first-ever Rx per group (any patient, no date floor)
first_rx = rx.groupBy("PROVIDER_ID", "GRP").agg(F.min("SVC_DT").alias("FIRST_RX_DT"))
# AD3: HCP's first brand Entresto Rx
first_brand_entresto = (rx.filter(F.col("PRODUCT_NAME").isin(ENTRESTO_BRAND_PRODUCTS))
                          .groupBy("PROVIDER_ID").agg(F.min("SVC_DT").alias("FIRST_BRAND_DT")))

In [0]:
min_gaps = (gaps_m.join(F.broadcast(coh), F.col("RX_M") <= F.col("LOOKBACK_END_M"))
            .groupBy("PROVIDER_ID", "cohort_id", "GRP")
            .agg(F.min("GAP_SAME").alias("F1"), F.min("GAP_ANY").alias("F3")))

appr_col = F.create_map(*[x for g, d in APPROVAL_DATES.items() for x in (F.lit(g), F.lit(d).cast("date"))])
adopt = (first_rx.filter(F.col("GRP").isin(list(APPROVAL_DATES)))
         .join(F.broadcast(coh), F.col("FIRST_RX_DT") <= F.last_day("LOOKBACK_END_M"))
         .select("PROVIDER_ID", "cohort_id", "GRP", F.datediff("FIRST_RX_DT", appr_col[F.col("GRP")]).alias("F2")))

long = min_gaps.join(adopt, ["PROVIDER_ID", "cohort_id", "GRP"], "full")
aggs = []
for g in DRUG_GROUPS:
    aggs.append(F.min(F.when(F.col("GRP") == g, F.col("F1"))).alias(f1(g)))
    if g in APPROVAL_DATES:
        aggs.append(F.min(F.when(F.col("GRP") == g, F.col("F2"))).alias(f2(g)))
    aggs.append(F.min(F.when(F.col("GRP") == g, F.col("F3"))).alias(f3(g)))
wide = long.groupBy("PROVIDER_ID", "cohort_id").agg(*aggs)

# AD3: brand adopter = first brand Entresto Rx before the LOE date and on or before the cohort cutoff
flag = (first_brand_entresto.join(F.broadcast(coh), F.col("FIRST_BRAND_DT") <= F.last_day("LOOKBACK_END_M"))
        .filter(F.col("FIRST_BRAND_DT") < F.lit(ENTRESTO_LOE_DATE).cast("date"))
        .select("PROVIDER_ID", "cohort_id", F.lit(1).alias(FLAG_COL)))

adoption_features = (hcp_universe.crossJoin(F.broadcast(coh))
    .join(wide, ["PROVIDER_ID", "cohort_id"], "left")
    .join(flag, ["PROVIDER_ID", "cohort_id"], "left")
    .fillna(0, subset=[FLAG_COL])
    .withColumnRenamed("LOOKBACK_END_M", "SVC_DT_M")
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
    .withColumn("PULL_DT", F.lit(dt.today().strftime("%Y_%m_%d")))
    .select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID",
            *[F.col(c).cast("int").alias(c) for c in FEATURE_COLS], "PULL_DT"))

os.makedirs(HCP_FEATURES_OUT_31, exist_ok=True)
adoption_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUTPUT_PATH)
afe = spark.read.format("delta").load(OUTPUT_PATH)
print(f"Saved {afe.count():,} rows, {len(afe.columns)} columns -> {OUTPUT_PATH}")

In [0]:
n_rows = afe.count()
assert n_rows == n_hcp * len(cohorts), (n_rows, n_hcp * len(cohorts))
assert afe.select("HCP_COHORT_ID").distinct().count() == n_rows, "HCP_COHORT_ID not unique"
assert afe.join(hcp_universe, "PROVIDER_ID", "left_anti").count() == 0
min_cols = [c for c in DAY_COLS if c.startswith("MIN_DAYS")]
neg = afe.filter(" OR ".join(f"{c} < 0" for c in min_cols)).count()

# Later cohorts see more claims: a min can only fall, #2 never changes once set, the flag never goes back to 0
wc = Window.partitionBy("PROVIDER_ID").orderBy("SVC_DT_M")
prev = afe.select("PROVIDER_ID", "SVC_DT_M", *[F.lag(c).over(wc).alias(f"p_{c}") for c in FEATURE_COLS], *FEATURE_COLS)
bad_min = prev.filter(" OR ".join(f"(p_{c} IS NOT NULL AND ({c} IS NULL OR {c} > p_{c}))" for c in min_cols)).count()
bad_adopt = prev.filter(" OR ".join(f"(p_{c} IS NOT NULL AND ({c} IS NULL OR {c} != p_{c}))" for c in DAY_COLS if c.startswith("HCP_TIME"))).count()
bad_flag = prev.filter(f"p_{FLAG_COL} = 1 AND {FLAG_COL} = 0").count()
print(f"rows {n_rows:,} | negative mins {neg} | min rose/vanished {bad_min} | #2 changed {bad_adopt} | flag 1->0 {bad_flag}")
assert neg == 0 and bad_min == 0 and bad_adopt == 0 and bad_flag == 0

# Independent check: recompute the last cohort directly from claims with SVC_DT <= cutoff
c = cohorts.iloc[-1]
cut = F.last_day(F.lit(str(c.lookback_end_date)[:10]).cast("date"))
dxc, rxc = dx.filter(F.col("SVC_DT") <= cut), rx.filter(F.col("SVC_DT") <= cut)
cols = []
for g in DRUG_GROUPS:
    fl = DATE_FLOORS.get(g)
    dxg = dxc if fl is None else dxc.filter(F.col("SVC_DT") >= F.lit(fl).cast("date"))
    rxg = rxc.filter(F.col("GRP") == g)
    rxg_f = rxg if fl is None else rxg.filter(F.col("SVC_DT") >= F.lit(fl).cast("date"))
    pr = rxg_f.groupBy("PROVIDER_ID", "PATIENT_ID").agg(F.min("SVC_DT").alias("R"))
    cols.append(pr.join(dxg.groupBy("PROVIDER_ID", "PATIENT_ID").agg(F.min("SVC_DT").alias("D")), ["PROVIDER_ID", "PATIENT_ID"])
                  .withColumn("G", F.datediff("R", "D")).filter("G >= 0").groupBy("PROVIDER_ID").agg(F.min("G").alias(f1(g))))
    cols.append(pr.join(dxg.groupBy("PATIENT_ID").agg(F.min("SVC_DT").alias("D")), "PATIENT_ID")
                  .withColumn("G", F.datediff("R", "D")).filter("G >= 0").groupBy("PROVIDER_ID").agg(F.min("G").alias(f3(g))))
    if g in APPROVAL_DATES:
        cols.append(rxg.groupBy("PROVIDER_ID").agg(F.datediff(F.min("SVC_DT"), F.lit(APPROVAL_DATES[g]).cast("date")).alias(f2(g))))
cols.append(rxc.filter(F.col("PRODUCT_NAME").isin(ENTRESTO_BRAND_PRODUCTS)).groupBy("PROVIDER_ID")
               .agg((F.min("SVC_DT") < F.lit(ENTRESTO_LOE_DATE).cast("date")).cast("int").alias(FLAG_COL)))
chk = hcp_universe
for x in cols:
    chk = chk.join(x, "PROVIDER_ID", "left")
chk = chk.fillna(0, subset=[FLAG_COL]).select("PROVIDER_ID", *[F.col(x).alias(f"chk_{x}") for x in FEATURE_COLS])
cmp = afe.filter(F.col("cohort_id") == c.cohort_id).join(chk, "PROVIDER_ID")
mism = cmp.filter(" OR ".join(f"NOT ({x} <=> chk_{x})" for x in FEATURE_COLS)).count()
print(f"{c.cohort_id} (cutoff {c.lookback_end_date}): mismatches vs direct recompute = {mism}")
assert mism == 0

In [0]:
last_df = afe.filter(F.col("cohort_id") == cohorts.cohort_id.iloc[-1])
stats = last_df.agg(*[F.count(c).alias(f"n_{c}") for c in DAY_COLS],
                    *[F.percentile_approx(c, 0.5).alias(f"med_{c}") for c in DAY_COLS],
                    F.sum(FLAG_COL).alias("n_flag")).collect()[0]
display(pd.DataFrame({"feature": DAY_COLS,
                      "pct_universe_with_value": [builtins.round(100 * stats[f"n_{c}"] / n_hcp, 1) for c in DAY_COLS],
                      "median_days": [stats[f"med_{c}"] for c in DAY_COLS]}))
print(f"{FLAG_COL} = 1 for {stats['n_flag']:,} HCPs ({builtins.round(100 * stats['n_flag'] / n_hcp, 1)}%)")

In [0]:
# Output table: one row per HCP x claims cohort (key HCP_COHORT_ID)
# Reads the saved output, so this cell and the next only need 'Paths and parameters' and 'Load claims cohorts and HCP universe' run first
afe = spark.read.format("delta").load(OUTPUT_PATH)
FEATURE_COLS = [c for c in afe.columns if c not in {"PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID", "PULL_DT"}]
display(afe.select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID", *FEATURE_COLS))

In [0]:
# Fill rate per feature: % of rows with a value (not null) and % with a non-zero value,
# over all cohorts and for the latest cohort
latest = cohorts.cohort_id.iloc[-1]

def _fill(df):
    n = df.count()
    r = df.agg(*[F.count(c).alias(f"nn_{c}") for c in FEATURE_COLS],
               *[F.count(F.when(F.col(c) != 0, 1)).alias(f"nz_{c}") for c in FEATURE_COLS]).collect()[0]
    return ([builtins.round(100 * r[f"nn_{c}"] / n, 2) for c in FEATURE_COLS],
            [builtins.round(100 * r[f"nz_{c}"] / n, 2) for c in FEATURE_COLS])

nn_all, nz_all = _fill(afe)
nn_last, nz_last = _fill(afe.filter(F.col("cohort_id") == latest))
display(pd.DataFrame({"feature": FEATURE_COLS,
                      "pct_filled_all_cohorts": nn_all, "pct_non_zero_all_cohorts": nz_all,
                      f"pct_filled_{latest}": nn_last, f"pct_non_zero_{latest}": nz_last}))

## Feature dictionary, business rules and business logic

**Features created (15).** `{DRUG}` = `INVOKANA`, `FARXIGA`, `JARDIANCE`, `STEROIDAL_MRA`, `ENTRESTO`.

| # | Feature | Definition | Drugs |
| --- | --- | --- | --- |
| 1-5 | `MIN_DAYS_TO_PRESCRIBE_{DRUG}_POST_CKD_DIAGNOSIS` | Smallest number of days, over the HCP's patients, from the patient's first CKD Dx **by this HCP** to the patient's first Rx of the drug from this HCP (NBA 3.0 `TIME_TO_ADOPT_{DRUG}` renamed) | all 5 |
| 6-9 | `HCP_TIME_TO_ADOPT_{DRUG}` | Days from the drug's FDA approval to the HCP's first Rx of the drug (any patient) | Invokana, Farxiga, Jardiance, Entresto |
| 10-14 | `MIN_DAYS_TO_PRESCRIBE_{DRUG}_POST_FIRST_CKD_DIAGNOSIS_ANY_HCP` | Same as 1-5, but from the patient's first CKD Dx by **any** provider | all 5 |
| 15 | `ENTRESTO_BRAND_ADOPTER_FLAG` | 1 if the HCP's first brand Entresto Rx is before the Entresto LOE date (2025-07-01) and on or before the cohort cutoff, else 0 | Entresto |

**Business rules**

1. **Drug groups (`Rx_01_new` `PRODUCT_NAME`):** Invokana; Farxiga; Jardiance; steroidal MRA = SPIRONOLACTONE + EPLERENONE (SPIRONOLACTONE/HYDROCHLOR not included); Entresto = ENTRESTO (brand) + SACUBITRIL/VALSARTAN and SACUBITRIL/VALSART (generic, from 2024-11-27). The brand-adopter flag uses brand only.
2. **CKD diagnosis:** NBA 3.1 Dx event list (`dx_input_claims_all_events_features_input`), codes N18.1, N18.2, N18.3, N18.30, N18.31, N18.32, N18.4, N18.5, N18.9, E11.21, E11.22, E11.29, E10.21, E10.22, E10.29, I12.9, I13.0, I13.10 (CKD), N18.6, Z99.2, I12.0, I13.11, I13.2 (ESRD / dialysis) and D63.1 (anemia in CKD). Replaces NBA 3.0 `Dx_03`, which has almost no stage 3 claims (V2).
3. **Date floors (as NBA 3.0):** Farxiga Dx and Rx on or after 2021-04-01 (CKD label); Invokana on or after 2019-10-01; other groups full history (data starts 2021-01-02).
4. **FDA approval dates** (confirm with Medical): Invokana 2013-03-29, Farxiga 2014-01-08, Jardiance 2014-08-01, Entresto 2015-07-07. Not built for steroidal MRA (no single approval date). These dates are before the data start, so HCPs who adopted before 2021 look like 2021 adopters.
5. **Negative gaps dropped:** a patient whose first CKD Dx is after the first Rx does not count.
6. **Rejected claims included** (as NBA 3.0).
7. **No future data:** each cohort uses claims up to the end of its `lookback_end_date` month.
8. **Null policy:** `MIN_DAYS_*` and `HCP_TIME_*` are empty when there is no qualifying patient / Rx by the cutoff (empty and 0 mean different things). The flag is 0 when the HCP did not write brand Entresto before LOE.
9. **Universe:** every HCP in `HCP_Universe_without_call_hcps` gets one row per claims cohort. The Dx event list and `Rx_01_new` are pulled on the same universe, so "any provider" means any universe provider.

**Business logic (steps)**

1. Read the CKD claims from the Dx event list (all providers; print the patient count vs `Dx_03`) and `Rx_01_new` (universe HCPs, the 5 drug groups); check every group is present and print the Entresto products found.
2. Per HCP x patient x group: first Rx date (after the date floor). Per HCP x patient and per patient: first CKD Dx date (after the floor). Gap = Rx date - CKD date, for the same HCP and for any provider.
3. Keep the smallest non-negative gap per HCP, group and Rx month. Per HCP and group: first-ever Rx date; per HCP: first brand Entresto date.
4. For each cohort: min gap over Rx months <= the cutoff (#1, #3), days from approval to the first Rx if that Rx is <= the cutoff (#2), and the brand-adopter flag.
5. Pivot to one column per feature, cross-join the universe with all cohorts, add `HCP_COHORT_ID` and write Delta to `hcp_features/adoption_features`.
6. Validate: row count, unique keys, no negative mins, values move in the right direction across cohorts (mins only fall, #2 never changes once set, the flag never goes back to 0) and a direct recompute of the latest cohort from claims capped at the cutoff.